# Mojibake: Encoding and Decoding Demo

This notebook demonstrates three ideas:

1. Text must be **encoded** into bytes before storage or transmission.
2. Bytes must be **decoded** with the matching encoding to recover the original text.
3. Mojibake appears when the same bytes are decoded with the wrong character encoding.

The main example uses the Chinese text `你好` because its UTF-8 byte sequence can be misread as GBK and displayed as `浣犲ソ`.


In [1]:
# Original text used in the classroom demo
text = "你好"

print("Original text:", text)


Original text: 你好


## 1. Correct encoding and decoding

The sender and receiver use the same rule: UTF-8.


In [2]:
# Encode the text into bytes using UTF-8
utf8_bytes = text.encode("utf-8")

# Decode the same bytes using UTF-8
decoded_text = utf8_bytes.decode("utf-8")

print("Original text :", text)
print("UTF-8 bytes   :", utf8_bytes)
print("Hex bytes     :", utf8_bytes.hex(" ").upper())
print("Decoded text  :", decoded_text)
print("Match         :", text == decoded_text)


Original text : 你好
UTF-8 bytes   : b'\xe4\xbd\xa0\xe5\xa5\xbd'
Hex bytes     : E4 BD A0 E5 A5 BD
Decoded text  : 你好
Match         : True


## 2. Create mojibake intentionally

The bytes are still correct.  
The problem is that the receiver uses the wrong decoding rule.

Here, UTF-8 bytes are incorrectly decoded as GBK.


In [3]:
# Encode correctly with UTF-8
correct_bytes = text.encode("utf-8")

# Decode incorrectly with GBK
mojibake_text = correct_bytes.decode("gbk")

print("Original text       :", text)
print("Stored byte sequence:", correct_bytes.hex(" ").upper())
print("Correct encoding    : UTF-8")
print("Wrong decoding      : GBK")
print("Displayed result    :", mojibake_text)


Original text       : 你好
Stored byte sequence: E4 BD A0 E5 A5 BD
Correct encoding    : UTF-8
Wrong decoding      : GBK
Displayed result    : 浣犲ソ


## 3. Correct vs. incorrect interpretation

The byte sequence does not change.  
Only the decoding rule changes.


In [4]:
# Show the same bytes interpreted in two different ways
same_bytes = text.encode("utf-8")

correct_result = same_bytes.decode("utf-8")
wrong_result = same_bytes.decode("gbk")

print("Same bytes          :", same_bytes.hex(" ").upper())
print("Decode as UTF-8     :", correct_result)
print("Decode as GBK       :", wrong_result)


Same bytes          : E4 BD A0 E5 A5 BD
Decode as UTF-8     : 你好
Decode as GBK       : 浣犲ソ


## 4. Inspect each character and its bytes

This cell makes the difference between a **character** and its **byte representation** more visible.


In [5]:
def show_character_bytes(value, encoding="utf-8"):
    # Print each character and its encoded byte sequence
    for character in value:
        encoded = character.encode(encoding)
        print(
            f"Character: {character} | "
            f"Encoding: {encoding} | "
            f"Bytes: {encoded.hex(' ').upper()} | "
            f"Byte count: {len(encoded)}"
        )

show_character_bytes("A你", "utf-8")


Character: A | Encoding: utf-8 | Bytes: 41 | Byte count: 1
Character: 你 | Encoding: utf-8 | Bytes: E4 BD A0 | Byte count: 3


## 5. Compare English and Chinese text

ASCII characters are represented by the same byte values inside UTF-8.  
This is one reason English text often survives encoding mismatches more easily than non-ASCII text.


In [6]:
samples = ["A", "Hello", "你", "你好"]

for sample in samples:
    encoded = sample.encode("utf-8")
    print("-" * 60)
    print("Text       :", sample)
    print("UTF-8 hex :", encoded.hex(" ").upper())
    print("Byte count :", len(encoded))


------------------------------------------------------------
Text       : A
UTF-8 hex : 41
Byte count : 1
------------------------------------------------------------
Text       : Hello
UTF-8 hex : 48 65 6C 6C 6F
Byte count : 5
------------------------------------------------------------
Text       : 你
UTF-8 hex : E4 BD A0
Byte count : 3
------------------------------------------------------------
Text       : 你好
UTF-8 hex : E4 BD A0 E5 A5 BD
Byte count : 6


## 6. A reusable mojibake simulator

Change the text, source encoding, and decoding encoding to test other cases.


In [7]:
def simulate_mojibake(text, source_encoding, wrong_decoding):
    # Step 1: Convert text into bytes using the original encoding
    raw_bytes = text.encode(source_encoding)

    # Step 2: Interpret the same bytes using a different encoding
    try:
        wrong_text = raw_bytes.decode(wrong_decoding)
    except UnicodeDecodeError as error:
        wrong_text = f"<Decoding failed: {error}>"

    print("Original text     :", text)
    print("Source encoding   :", source_encoding)
    print("Raw bytes         :", raw_bytes.hex(" ").upper())
    print("Decoding rule     :", wrong_decoding)
    print("Displayed result  :", wrong_text)

    return wrong_text

simulate_mojibake("你好", "utf-8", "gbk")


Original text     : 你好
Source encoding   : utf-8
Raw bytes         : E4 BD A0 E5 A5 BD
Decoding rule     : gbk
Displayed result  : 浣犲ソ


'浣犲ソ'

## 7. Recover the original text from this specific mojibake

For this exact scenario:

- original bytes were UTF-8,
- those bytes were mistakenly decoded as GBK,
- so recovery reverses the wrong decoding first and then decodes correctly.

This works only when the mojibake process has not already destroyed or replaced byte information.


In [ ]:
mojibake = "浣犲ソ"

# Reverse the wrong GBK decoding to recover the original byte sequence
recovered_bytes = mojibake.encode("gbk")

# Decode the recovered bytes correctly as UTF-8
recovered_text = recovered_bytes.decode("utf-8")

print("Mojibake text  :", mojibake)
print("Recovered bytes:", recovered_bytes.hex(" ").upper())
print("Recovered text :", recovered_text)


## Key conclusion

**Mojibake does not necessarily mean that the stored data is damaged.**

A common cause is:

> **Encoding rule used to create the bytes ≠ decoding rule used to interpret the bytes**

In the classroom example:

`你好`  
→ UTF-8 encoding  
→ `E4 BD A0 E5 A5 BD`  
→ incorrectly decoded as GBK  
→ `浣犲ソ`
